# 1. Plan normal restaurant service

Goals: identify preconditions and effects, inspect a minimum-cost plan, and validate execution. One tray holds one meal. Only travel drains energy; total action cost is not elapsed time.

These are symbolic planning exercises. Gazebo is a separate static scene; these cells do not command a robot. Run cells in order with Python 3.

In [ ]:
from pathlib import Path
import sys
# Works from the student folder, solutions folder, or notebook server root.
base = Path.cwd().resolve()
for candidate in (base, *base.parents, base / 'jupyter_notebooks'):
    if (candidate / 'robo_restaurant' / 'planning.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError('Open this notebook inside the repository')
from dataclasses import replace
from robo_restaurant.planning import State, Action, solve, execute, successors, goal


## Read the state and actions

`State` is an immutable snapshot: actions produce a new state instead of changing the old one. `State()` creates the initial situation below. Use `replace(state, field=value)` to describe a changed situation in later exercises.

| Field | Meaning | Initial value |
| --- | --- | --- |
| `location` | Waiter position: `dock`, `kitchen`, `table_1` or `table_2` | `'dock'` |
| `battery` | Available travel energy, from 0 to 8 | `8` |
| `carried` | Meal index on the one-meal tray: `-1` for empty, `0` for customer 1, `1` for customer 2 | `-1` |
| `meals` | Tuple for customers 1 and 2; each progresses through `ordered`, `ready`, `carried`, `served` | `('ordered', 'ordered')` |
| `cooker_ok` | Whether preparation is possible | `True` |

An `Action` contains `name` and `target`. For example, `Action('move', 'kitchen')` names a destination, while `Action('prepare', '1')` names customer 1. Customer targets are strings `'1'` and `'2'`; the `carried` field uses zero-based integer indices. `Action('charge')` and `Action('repair')` have an empty target.

| Action | Preconditions and effects | Action cost |
| --- | --- | --- |
| `move` | Follow a graph edge with enough battery; update location and subtract travel energy. Edges are dock–kitchen, dock–table_1, table_1–table_2 and kitchen–table_2, in both directions. | 2 per edge |
| `prepare` | At the kitchen, cooker working, selected meal ordered; mark it ready. | 3 |
| `pick_up` | At the kitchen, tray empty, selected meal ready; put it on the tray. | 1 |
| `serve` | At the selected customer's table carrying their meal; mark it served and empty the tray. | 1 |
| `charge` | At the dock with battery below 8; restore battery to 8. | 3 |
| `repair` | At the kitchen with a broken cooker and an empty tray; restore the cooker. | 5 |

In [ ]:
example_state = State()
print('Initial snapshot:', example_state)
print('Example action:', Action('move', 'kitchen'))

## Inspect one legal transition

`successors(state)` yields only actions whose preconditions hold in that state. Each item is a tuple `(action, next_state, cost)`: the action you may choose, the resulting snapshot, and that action's cost. Calling it does not change `state`. An action missing from this list is illegal in that state.

At the dock with a full battery, the waiter can move to the kitchen or table 1. Inspect the available transitions, then select just the move to the kitchen:

In [ ]:
available = list(successors(example_state))
for action, next_state, step_cost in available:
    print(action, '->', next_state, 'action cost:', step_cost)

move_action, at_kitchen, move_cost = next(
    transition for transition in available
    if transition[0] == Action('move', 'kitchen')
)
print('Location:', example_state.location, '->', at_kitchen.location)
print('Battery:', example_state.battery, '->', at_kitchen.battery)
print('Action cost:', move_cost)
assert example_state.location == 'dock' and example_state.battery == 8
assert at_kitchen.location == 'kitchen' and at_kitchen.battery == 6

## Distinguish action cost, energy and the goal

Only `move` consumes battery in this model. All actions have a cost, and `solve` minimizes their sum. Costs are abstract planning weights, not elapsed time. A move happens to cost the same number as the energy it consumes, but preparing a meal costs 3 while consuming no battery; charging costs 3 while increasing battery.

From the kitchen snapshot, inspect one preparation transition. It changes the first meal from `ordered` to `ready`, keeps location and battery unchanged, and adds 3 to the plan cost. These two example actions therefore cost 5 while consuming 2 battery units.

`goal(state)` is true only when **both meals are served and the waiter is back at the dock**. It does not require a full battery. A legal transition or a legal partial plan need not reach this goal.

In [ ]:
prepare_action, after_prepare, prepare_cost = next(
    transition for transition in successors(at_kitchen)
    if transition[0] == Action('prepare', '1')
)
print(prepare_action)
print('Meals:', at_kitchen.meals, '->', after_prepare.meals)
print('Location:', at_kitchen.location, '->', after_prepare.location)
print('Battery:', at_kitchen.battery, '->', after_prepare.battery)
print('Preparation cost:', prepare_cost)
print('Two-action cost:', move_cost + prepare_cost)
print('Battery consumed:', example_state.battery - after_prepare.battery)
print('Goal reached:', goal(after_prepare))
assert after_prepare.meals == ('ready', 'ordered')
assert after_prepare.location == at_kitchen.location
assert after_prepare.battery == at_kitchen.battery
assert move_cost + prepare_cost == 5 and not goal(after_prepare)

## Demonstration

In [ ]:
initial = State()
plan, cost = solve(initial)
print('Optimal cost:', cost)
for action in plan:
    print(action)
assert cost == 25

## Your task

Implement `audit_plan(initial, plan)` returning `(final_state, total_cost)`. Use legal transitions from `successors`; reject an illegal action with `ValueError`. Do not call `solve`: you are validating a supplied plan. A partial plan may be legal without reaching the goal.

In [ ]:
def audit_plan(initial, plan):
    # TODO: replace this body.
    raise NotImplementedError


## Check your answer

The starter runs without errors but reports an unfinished task. After implementing it, this cell must report PASS.

In [ ]:
try:
    final, actual = audit_plan(initial, plan)
    assert goal(final) and actual == 25
    assert audit_plan(initial, []) == (initial, 0)
    try:
        audit_plan(initial, [Action('serve', '1')])
    except ValueError:
        pass
    else:
        raise AssertionError('Illegal serving must be rejected')
except NotImplementedError:
    print("TODO: implement the exercise; this is not a pass")
else:
    print("PASS")

## Discussion

Why does the plan charge even with an initially full battery? List the preconditions of pickup and serving. Why is a valid partial plan different from a completed goal?

[Next exercise: Low battery](<13. Robo Restaurant - Low Battery.ipynb>)